In [1]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="Qwen/Qwen3-8B", base_url="https://api.siliconflow.cn")

from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict


agent = create_agent(
    model,
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "你好啊"}],
        "user_id": "user_123",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}})

print(result)

{'messages': [HumanMessage(content='你好啊', additional_kwargs={}, response_metadata={}, id='e85bce71-8728-4cc4-aff4-2eba89aa1e22'), AIMessage(content='你好！我是Qwen，一个由通义实验室开发的超大规模语言模型。有什么我可以帮助你的吗？比如回答问题、创作文字，或者进行其他有趣的对话？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 14, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '019b00b6a3b4e3ba197c05b7353c4b47', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a6405579-1586-4541-bcfd-ed027ceeecae-0', usage_metadata={'input_tokens': 14, 'output_tokens': 38, 'total_tokens': 52, 'input_token_details': {}, 'output_token_details': {'reasoning': 0}})], 'user_id': 'user_123', 'preferences': {'theme': 'dark'}}


##裁剪消息

In [4]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是林水"}, config)
agent.invoke({"messages": "写一首五言绝句"}, config)
agent.invoke({"messages": "写一首七言绝句"}, config)
final_response = agent.invoke({"messages": "我的名字是什么"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

你的名字是林水！😊 有什么我可以帮你的吗？


##删除消息

In [2]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove old messages to keep conversation manageable."""
    messages = state["messages"]
    if len(messages) > 2:
        # remove the earliest two messages
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None


agent = create_agent(
    model,
    system_prompt="Please be concise and to the point.",
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

for event in agent.stream(
        {"messages": [{"role": "user", "content": "hi! I'm bob"}]},
        config,
        stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

for event in agent.stream(
        {"messages": [{"role": "user", "content": "what's my name?"}]},
        config,
        stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

[('human', "hi! I'm bob")]
[('human', "hi! I'm bob"), ('ai', 'Hello, Bob! How can I assist you today?')]
[('human', "hi! I'm bob"), ('ai', 'Hello, Bob! How can I assist you today?'), ('human', "what's my name?")]
[('human', "hi! I'm bob"), ('ai', 'Hello, Bob! How can I assist you today?'), ('human', "what's my name?"), ('ai', 'Your name is Bob.')]
[('human', "what's my name?"), ('ai', 'Your name is Bob.')]


##总结消息


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

checkpointer = InMemorySaver()

agent = create_agent(
    model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 500),
            keep=("tokens", 200)
        )
    ],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": "你好，我是林水"}, config)
agent.invoke({"messages": "英雄联盟是什么"}, config)
agent.invoke({"messages": "云顶之奕是什么"}, config)
final_response = agent.invoke({"messages": "我是谁"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

你好！你可能是《英雄联盟》或《云顶之奕》的玩家，也有可能是你第一次了解到这两款游戏。无论你是谁，我都很乐意帮助你了解更多关于《英雄联盟》和《云顶之奕》的信息，或者解答你在游戏中的任何疑问。你可以告诉我你更感兴趣的是哪一款游戏，或者你想了解什么内容？😊


## 工具中写入短期记忆

In [8]:
from langchain.tools import tool, ToolRuntime
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command
from pydantic import BaseModel


class CustomState(AgentState):
    user_name: str


class CustomContext(BaseModel):
    user_id: str


@tool
def update_user_info(
        runtime: ToolRuntime[CustomContext, CustomState],
) -> Command:
    """Look up and update user info."""
    user_id = runtime.context.user_id
    name = "John Smith" if user_id == "user_123" else "Unknown user"
    return Command(update={
        "user_name": name,
        # update the message history
        "messages": [
            ToolMessage(
                "Successfully looked up user information",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


@tool
def greet(
        runtime: ToolRuntime[CustomContext, CustomState]
) -> str | Command:
    """Use this to greet the user once you found their info."""
    user_name = runtime.state.get("user_name", None)
    if user_name is None:
        return Command(update={
            "messages": [
                ToolMessage(
                    "Please call the 'update_user_info' tool it will get and update the user's name.",
                    tool_call_id=runtime.tool_call_id
                )
            ]
        })
    return f"Hello {user_name}!"


agent = create_agent(
    model=model,
    tools=[update_user_info, greet],
    state_schema=CustomState,
    context_schema=CustomContext,
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "greet the user"}]},
    context=CustomContext(user_id="user_123"),
)
print(response)

{'messages': [HumanMessage(content='greet the user', additional_kwargs={}, response_metadata={}, id='0e4409a8-2b6e-4cc6-a747-34631aef74fb'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '019b00cf2a980470f24ad6241d83b5dc', 'function': {'arguments': '{}', 'name': 'greet'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 194, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '019b00cf2845c8825ab7b5a4fe3a3215', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--5af6d749-3c44-4e09-b2f5-9e9a894fdfb9-0', tool_calls=[{'name': 'greet', 'args': {}, 'id': '019b00cf2a980470f24ad6241d83b5dc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 194, 'output_tokens': 15